In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()  # Load environment variables from .env file

llm = ChatOpenAI(model="gpt-5.4-nano")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState) -> JokeState:

    topic = state['topic']

    # Generate a joke based on the topic
    joke_prompt = f"Tell me a funny joke about {topic}."
    joke_response = llm.invoke(joke_prompt).content

    return {'joke': joke_response}

def explain_joke(state: JokeState) -> JokeState:

    joke = state['joke']

    # Explain the joke
    explanation_prompt = f"Explain the following joke: {joke}"
    explanation_response = llm.invoke(explanation_prompt).content

    return {'explanation': explanation_response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {'configurable': {'thread_id': '1'}}

workflow.invoke({'topic':'pizza'},config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get kicked off the sports team?  \n\nBecause it kept **letting everyone down—one slice at a time!** 🍕😄',
 'explanation': 'The joke is a wordplay on the phrase **“letting everyone down.”**\n\n- A pizza being **“let off/kicked off the team”** sounds like a sports reason.\n- The punchline says it kept **letting people down “one slice at a time”**—meaning it’s **falling apart or failing gradually**, slice by slice.\n- It also plays on the idea that the pizza is the “player,” and each slice represents another “failure,” like poor performance in games.'}

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get kicked off the sports team?  \n\nBecause it kept **letting everyone down—one slice at a time!** 🍕😄', 'explanation': 'The joke is a wordplay on the phrase **“letting everyone down.”**\n\n- A pizza being **“let off/kicked off the team”** sounds like a sports reason.\n- The punchline says it kept **letting people down “one slice at a time”**—meaning it’s **falling apart or failing gradually**, slice by slice.\n- It also plays on the idea that the pizza is the “player,” and each slice represents another “failure,” like poor performance in games.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f4d-787f-67db-8002-39de28406297'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-06T13:14:07.832222+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f4d-6705-674f-8001-b3c357545330'}}, tasks=(), inte

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get kicked off the sports team?  \n\nBecause it kept **letting everyone down—one slice at a time!** 🍕😄', 'explanation': 'The joke is a wordplay on the phrase **“letting everyone down.”**\n\n- A pizza being **“let off/kicked off the team”** sounds like a sports reason.\n- The punchline says it kept **letting people down “one slice at a time”**—meaning it’s **falling apart or failing gradually**, slice by slice.\n- It also plays on the idea that the pizza is the “player,” and each slice represents another “failure,” like poor performance in games.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f4d-787f-67db-8002-39de28406297'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-06T13:14:07.832222+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f4d-6705-674f-8001-b3c357545330'}}, tasks=(), int

In [13]:
config2 = {'configurable': {'thread_id': '2'}}

workflow.invoke({'topic':'pasta'},config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta get promoted at work?  \n\nBecause it was always **ahead of the sauce**. 🍝😄',
 'explanation': 'The joke plays on a double meaning of **“ahead of the sauce.”**\n\n- **Literal/food sense:** Pasta is usually served with sauce on top, so the pasta being “ahead of the sauce” is a playful, visual idea.\n- **Workplace sense:** In office jokes, “being ahead” implies **working efficiently or getting promoted because you’re performing better** than others.\n\nSo the punchline is that the pasta earned a promotion because it was “ahead” of the sauce—mixing the workplace reason with a silly description of serving pasta.'}

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta get promoted at work?  \n\nBecause it was always **ahead of the sauce**. 🍝😄', 'explanation': 'The joke plays on a double meaning of **“ahead of the sauce.”**\n\n- **Literal/food sense:** Pasta is usually served with sauce on top, so the pasta being “ahead of the sauce” is a playful, visual idea.\n- **Workplace sense:** In office jokes, “being ahead” implies **working efficiently or getting promoted because you’re performing better** than others.\n\nSo the punchline is that the pasta earned a promotion because it was “ahead” of the sauce—mixing the workplace reason with a silly description of serving pasta.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f56-8941-603d-8002-6ae6357f0be9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-06T13:18:11.181091+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta get promoted at work?  \n\nBecause it was always **ahead of the sauce**. 🍝😄', 'explanation': 'The joke plays on a double meaning of **“ahead of the sauce.”**\n\n- **Literal/food sense:** Pasta is usually served with sauce on top, so the pasta being “ahead of the sauce” is a playful, visual idea.\n- **Workplace sense:** In office jokes, “being ahead” implies **working efficiently or getting promoted because you’re performing better** than others.\n\nSo the punchline is that the pasta earned a promotion because it was “ahead” of the sauce—mixing the workplace reason with a silly description of serving pasta.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f56-8941-603d-8002-6ae6357f0be9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-06T13:18:11.181091+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoin

In [19]:
workflow.get_state({"configurable":{"thread_id": "2", 'checkpoint_id': '1f1a9f56-6829-619d-8000-2c47a26796c2'}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1a9f56-6829-619d-8000-2c47a26796c2'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-06T13:18:07.711018+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f56-681f-6033-bfff-58c9197957cb'}}, tasks=(PregelTask(id='43c52177-c315-d652-1c56-41a59cfcd692', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pasta get promoted at work?  \n\nBecause it was always **ahead of the sauce**. 🍝😄'}),), interrupts=())

In [20]:
workflow.invoke(None, {"configurable":{"thread_id": "2", 'checkpoint_id': '1f1a9f56-6829-619d-8000-2c47a26796c2'}})

{'topic': 'pasta',
 'joke': 'Why did the pasta file a police report?  \n\nBecause it got *spaghetti-napped*! 🍝😄',
 'explanation': 'The joke is a pun on the word **“kidnapped.”**\n\n- **“Spaghetti-napped”** sounds like **“kidnapped.”**\n- The setup—“Why did the pasta file a police report?”—suggests something was taken or stolen.\n- The punchline says it was **spaghetti-napped**, meaning someone “napped” (kidnapped) the spaghetti.\n\nSo the humor comes from turning a common crime word into a playful pasta-themed version.'}

In [21]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta file a police report?  \n\nBecause it got *spaghetti-napped*! 🍝😄', 'explanation': 'The joke is a pun on the word **“kidnapped.”**\n\n- **“Spaghetti-napped”** sounds like **“kidnapped.”**\n- The setup—“Why did the pasta file a police report?”—suggests something was taken or stolen.\n- The punchline says it was **spaghetti-napped**, meaning someone “napped” (kidnapped) the spaghetti.\n\nSo the humor comes from turning a common crime word into a playful pasta-themed version.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f87-ff1f-6c23-8003-e7ce4ab540fb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-06T13:40:18.874390+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f87-ebd5-62b0-8002-b6df6de58424'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the 

In [23]:
workflow.update_state({"configurable":{"thread_id": "2", 'checkpoint_id': '1f1a9f56-6829-619d-8000-2c47a26796c2', 'checkpoint_ns': ""}}, {'topic': 'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a9f8e-f006-6d28-8001-881af9f60753'}}

In [24]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f8e-f006-6d28-8001-881af9f60753'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-06T13:43:25.196188+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f56-6829-619d-8000-2c47a26796c2'}}, tasks=(PregelTask(id='6a3bd2bd-e7a4-94b8-a6ac-6fd3097e603c', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta file a police report?  \n\nBecause it got *spaghetti-napped*! 🍝😄', 'explanation': 'The joke is a pun on the word **“kidnapped.”**\n\n- **“Spaghetti-napped”** sounds like **“kidnapped.”**\n- The setup—“Why did the pasta file a police report?”—suggests something was taken or stolen.\n- The punchline says it was *

In [26]:
workflow.invoke(None, {"configurable":{"thread_id": "2", 'checkpoint_id': '1f1a9f8e-f006-6d28-8001-881af9f60753'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party?\n\nBecause it heard the snacks were going to be *up for grabs*! 😄',
 'explanation': 'The joke plays on a pun about the word **“up for grabs.”**\n\n- **“Up for grabs”** is a common phrase meaning *available to be taken/claimed*.\n- The samosa is bringing a **ladder**, which makes you think of something being **physically “up” somewhere**.\n- So the joke suggests: the samosa heard the snacks were “up for grabs” and thought, literally, they’d be **up high**, so it brought a ladder to reach them.\n\nIt’s basically a *literal interpretation* of the idiom. 😄'}

In [27]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party?\n\nBecause it heard the snacks were going to be *up for grabs*! 😄', 'explanation': 'The joke plays on a pun about the word **“up for grabs.”**\n\n- **“Up for grabs”** is a common phrase meaning *available to be taken/claimed*.\n- The samosa is bringing a **ladder**, which makes you think of something being **physically “up” somewhere**.\n- So the joke suggests: the samosa heard the snacks were “up for grabs” and thought, literally, they’d be **up high**, so it brought a ladder to reach them.\n\nIt’s basically a *literal interpretation* of the idiom. 😄'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f92-9225-6a20-8003-f1407a53adf0'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-06T13:45:02.726390+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9f92-7d94-60b